# k-Nearest Neighbors (kNN) Masterclass

**Complete guide from basics to expert level!**

## What You'll Master:
1. Core Algorithm & Intuition
2. Distance Metrics Deep Dive
3. Building from Scratch
4. Choosing Optimal k
5. Feature Scaling (CRITICAL!)
6. Real Applications
7. Best Practices & Pitfalls

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_iris, load_digits
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import Counter
import pandas as pd

sns.set_style('whitegrid')
np.random.seed(42)
print('✅ Ready to learn kNN!')

## 1. The Intuition

**Core Idea:** You are similar to your neighbors

### Real-World Example
Estimating house prices:
- Find 5 similar houses nearby
- Average their prices
- That's your estimate!

**kNN does exactly this for any prediction task!**

In [ ]:
# Visual example
np.random.seed(42)

# Create two classes
class_0 = np.random.randn(20, 2) * 0.5 + [2, 2]  # Blue cluster
class_1 = np.random.randn(20, 2) * 0.5 + [5, 5]  # Red cluster

X_train = np.vstack([class_0, class_1])
y_train = np.array([0]*20 + [1]*20)

# New point to classify
new_point = np.array([[3.5, 3.5]])

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(class_0[:, 0], class_0[:, 1], c='blue', label='Class 0', s=100, alpha=0.6)
plt.scatter(class_1[:, 0], class_1[:, 1], c='red', label='Class 1', s=100, alpha=0.6)
plt.scatter(new_point[0,0], new_point[0,1], c='green', marker='*', s=500, 
            edgecolors='black', linewidths=2, label='New Point', zorder=5)

# Find 3 nearest
distances = np.sqrt(np.sum((X_train - new_point)**2, axis=1))
nearest_3 = np.argsort(distances)[:3]

# Draw connections
for idx in nearest_3:
    plt.plot([new_point[0,0], X_train[idx,0]], 
             [new_point[0,1], X_train[idx,1]], 'k--', alpha=0.5, linewidth=2)
    plt.scatter(X_train[idx,0], X_train[idx,1], s=200, 
                facecolors='none', edgecolors='yellow', linewidths=3)

votes = y_train[nearest_3]
prediction = np.bincount(votes).argmax()

plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title(f'kNN with k=3\nVotes: {list(votes)} → Prediction: Class {prediction}', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f'🎯 Predicted class: {prediction}')

## 2. The Algorithm (5 Steps)

```
For each test point:
  STEP 1: Calculate distance to ALL training points
  STEP 2: Sort by distance
  STEP 3: Select k nearest neighbors
  STEP 4: Vote (classification) or Average (regression)
  STEP 5: Return prediction
```

## 3. Distance Metrics

### Euclidean Distance (Most Common)

Formula: d = √[(x₁-x₂)² + (y₁-y₂)²]

**Think:** Straight-line distance

In [ ]:
# Euclidean distance example
point_A = np.array([1, 2])
point_B = np.array([4, 6])

euclidean = np.sqrt(np.sum((point_B - point_A)**2))
manhattan = np.sum(np.abs(point_B - point_A))

print('Distance Metrics:')
print(f'  Euclidean: {euclidean:.3f} (straight line)')
print(f'  Manhattan: {manhattan:.3f} (grid path)')

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(*point_A, s=200, c='blue', label='Point A', zorder=3)
plt.scatter(*point_B, s=200, c='red', label='Point B', zorder=3)
plt.plot([point_A[0], point_B[0]], [point_A[1], point_B[1]], 
         'g--', linewidth=2, label=f'Euclidean={euclidean:.2f}')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Euclidean Distance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

## 4. kNN from Scratch

Let's build it ourselves!

In [ ]:
class KNNFromScratch:
    def __init__(self, k=3):
        self.k = k
    
    def fit(self, X, y):
        '''Just store the data (lazy learning!)'''
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self
    
    def _predict_single(self, x):
        # Step 1: Calculate distances
        distances = [np.sqrt(np.sum((x - x_train)**2)) 
                    for x_train in self.X_train]
        
        # Step 2 & 3: Get k nearest
        k_indices = np.argsort(distances)[:self.k]
        
        # Step 4: Vote
        k_labels = self.y_train[k_indices]
        most_common = Counter(k_labels).most_common(1)
        
        # Step 5: Return
        return most_common[0][0]
    
    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])
    
    def score(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == y)

print('✅ kNN implemented from scratch!')

# Test it
X, y = make_classification(n_samples=100, n_features=2, n_redundant=0, 
                           n_informative=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

knn = KNNFromScratch(k=5)
knn.fit(X_train, y_train)
acc = knn.score(X_test, y_test)

print(f'Accuracy: {acc*100:.2f}%')
print('🎉 It works!')

## 5. Choosing k - CRITICAL!

### Bias-Variance Tradeoff

- **k=1:** Overfitting (too flexible)
- **k=N:** Underfitting (too simple)
- **k=moderate:** Just right! ✨

**Rule of thumb:** k ≈ √N

In [ ]:
# Find optimal k with cross-validation
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# IMPORTANT: Scale features!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Try different k values
k_range = range(1, 31)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    cv_scores.append(scores.mean())

best_k = k_range[np.argmax(cv_scores)]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, cv_scores, marker='o', linewidth=2)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k (Number of Neighbors)', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Finding Optimal k', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f'🎯 Best k: {best_k}')
print(f'CV Accuracy: {max(cv_scores)*100:.2f}%')
print(f'\nRule of thumb: √{len(X_train)} ≈ {int(np.sqrt(len(X_train)))}')

## 6. Feature Scaling - ABSOLUTELY CRITICAL! ⚠️

**kNN uses distance → feature scales matter HUGELY!**

In [ ]:
# Demonstrate importance of scaling
np.random.seed(42)

# Feature 1: small (0-10), Feature 2: HUGE (1000-10000)
n = 50
X_0 = np.column_stack([np.random.randn(n) + 3, np.random.randn(n)*500 + 3000])
X_1 = np.column_stack([np.random.randn(n) + 7, np.random.randn(n)*500 + 7000])

X = np.vstack([X_0, X_1])
y = np.array([0]*n + [1]*n)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print('Feature Ranges:')
print(f'  Feature 1: [{X[:,0].min():.1f}, {X[:,0].max():.1f}]')
print(f'  Feature 2: [{X[:,1].min():.1f}, {X[:,1].max():.1f}]')
print(f'\n⚠️ Feature 2 is 1000x larger!')

# WITHOUT scaling
knn_no = KNeighborsClassifier(n_neighbors=5)
knn_no.fit(X_train, y_train)
acc_no = knn_no.score(X_test, y_test)

# WITH scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

knn_yes = KNeighborsClassifier(n_neighbors=5)
knn_yes.fit(X_train_sc, y_train)
acc_yes = knn_yes.score(X_test_sc, y_test)

print(f'\nResults:')
print(f'  WITHOUT scaling: {acc_no*100:.2f}%')
print(f'  WITH scaling:    {acc_yes*100:.2f}%')
print(f'\n🚀 Improvement: +{(acc_yes-acc_no)*100:.1f}%!')
print(f'\n💡 ALWAYS scale features for kNN!')

## 7. Real Application: Handwritten Digits

In [ ]:
# Load digits dataset
digits = load_digits()
X, y = digits.data, digits.target

print(f'Dataset: {len(X)} images of digits (0-9)')
print(f'Each image: 8x8 pixels = 64 features')

# Show samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for idx, ax in enumerate(axes.flat):
    ax.imshow(digits.images[idx], cmap='gray')
    ax.set_title(f'Label: {y[idx]}')
    ax.axis('off')
plt.suptitle('Sample Handwritten Digits', fontsize=14)
plt.tight_layout()
plt.show()

# Train kNN
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)

test_acc = knn.score(X_test_scaled, y_test)
print(f'\n✅ Test Accuracy: {test_acc*100:.2f}%')

# Show predictions
y_pred = knn.predict(X_test_scaled[:10])

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for idx, ax in enumerate(axes.flat):
    ax.imshow(X_test[idx].reshape(8,8), cmap='gray')
    true, pred = y_test[idx], y_pred[idx]
    color = 'green' if true == pred else 'red'
    ax.set_title(f'T:{true} P:{pred}', color=color)
    ax.axis('off')
plt.suptitle('Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Complete Checklist

```
✅ PREPROCESSING:
  □ Scale features (StandardScaler) - CRITICAL!
  □ Handle missing values
  □ Remove irrelevant features

⚙️ HYPERPARAMETERS:
  □ Find k with cross-validation
  □ Use odd k (binary classification)
  □ Start with k ≈ √N
  □ Try different distance metrics

📊 EVALUATION:
  □ Use cross-validation
  □ Check confusion matrix
  □ Monitor train vs test accuracy
```

## 9. Summary

### ✅ Advantages
- Simple and intuitive
- No training (lazy learning)
- Works for classification & regression
- No assumptions about data

### ❌ Disadvantages
- Slow for large datasets
- Memory intensive
- **MUST scale features**
- Curse of dimensionality

### 💡 Key Takeaways
1. **ALWAYS scale features!**
2. Use cross-validation for k
3. Start with k ≈ √N
4. Euclidean usually works
5. Great for small-medium datasets

### 🎯 When to Use
- Small datasets (< 100k)
- Low dimensions (< 50 features)
- Need quick baseline
- Interpretability matters

---

**You've mastered kNN!** 🎉